# Kernels

In [21]:
import gpytorch
import math
import torch
from matplotlib import pyplot as plt
import numpy as np

# %matplotlib inline
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
# Training data is 100 points in [0,1] inclusive regularly spaced
train_x = torch.linspace(0, 1, 0)

# True function is sin(2*pi*x) with Gaussian noise
train_y = torch.sin(train_x * (2 * math.pi)) + torch.randn(train_x.size()) * math.sqrt(0.04)

In [ ]:
# We will use the simplest form of GP model, exact inference
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ExactGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ExactGPModel(train_x, train_y, likelihood)

In [ ]:
model.covar_module.base_kernel.parameters()

In [ ]:
# this is for running the notebook in our testing framework
import os
smoke_test = ('CI' in os.environ)
training_iter = 2 if smoke_test else 50


# Find optimal model hyperparameters
model.train()
likelihood.train()

# Use the adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1)  # Includes GaussianLikelihood parameters

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(train_x)
    # Calc loss and backprop gradients
    loss = - mll(output, train_y)
    loss.backward()
    print('Iter %d/%d - Loss: %.3f   lengthscale: %.3f   noise: %.3f' % (
        i + 1, training_iter, loss.item(),
        model.covar_module.base_kernel.lengthscale.item(),
        model.likelihood.noise.item()
    ))
    optimizer.step()

In [ ]:
# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()

# Test points are regularly spaced along [0,1]
# Make predictions by feeding model through likelihood
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    test_x = torch.linspace(0, 2, 51)
    f_preds = model(test_x)
    y_preds = likelihood(model(test_x))

In [ ]:
f_mean = f_preds.mean
f_var = f_preds.variance
f_covar = f_preds.covariance_matrix

In [ ]:
plt.imshow(f_covar.detach().numpy(), cmap = 'viridis')
plt.title("Posterior covariance matrix across larger domain")
# legend
plt.colorbar()
plt.show()

In [ ]:
with torch.no_grad():
    # Initialize plot
    f, ax = plt.subplots(1, 1, figsize = (8, 6))

    # Get upper and lower confidence bounds
    lower, upper = y_preds.confidence_region()
    # Plot training data as black stars
    ax.plot(train_x.numpy(), train_y.numpy(), 'k*')
    # Plot predictive means as blue line
    ax.plot(test_x.numpy(), y_preds.mean.numpy(), 'b')
    # Shade between the lower and upper confidence bounds
    ax.fill_between(test_x.numpy(), lower.numpy(), upper.numpy(), alpha = 0.5)
    ax.set_ylim([-3, 3])
    ax.legend(['Observed Data', 'Mean', 'Confidence'])

# 2 D

In [ ]:
n_train = 100

# Training data is 100 points in [0,1] inclusive regularly spaced
train_x = torch.linspace(0, 1, n_train)

mesh1, mesh2 = np.meshgrid(train_x, train_x)
# mesh == mesh2.T

In [ ]:
train_x_expldim = train_x.unsqueeze(0)
train_x_2D = torch.cat((train_x_expldim, train_x_expldim), dim = 0)
train_x_2D.shape

In [ ]:
tile = torch.tile(train_x_expldim, dims = (100, 1))

noise = 0 # 0.04
# train_y = torch.sin((train_x_2D * 0.5) + (train_x_2D[0] + train_x_2D[1] * (math.pi))
#            + torch.randn(train_x_2D.size()[1]) * math.sqrt(noise))

train_y = (torch.sin(torch.tensor((mesh1 * 4) + (mesh2 * 4)) + (tile * 4)) * 0.25)

In [ ]:
 fig = go.Figure(data = [go.Surface(
    z = train_y, 
    x = train_x_2D[0], 
    y = train_x_2D[1],
    opacity = 0.7
    )])

fig.update_layout(title = 'Surface Mass Balance (SMB) for Antarctica - Dec 2022',
                  width = 700, height = 700,
                  margin = dict(l = 65, r = 50, b = 65, t = 90)
                  )

fig.show()

## Scene data

In [3]:
hr_scene_bed_tensor = torch.load("./torch_data/scene_bed_tensor.pt")
hr_scene_sur_tensor = torch.load("./torch_data/scene_sur_tensor.pt")

In [19]:
print("HR tensor shape:", hr_scene_bed_tensor.unsqueeze(0).shape)

# Magnification factor should if divisor of shape
magnification_factor = 5

# Check remainder
if ((hr_scene_bed_tensor.shape[-1] % magnification_factor) != 0):
    print("ACHTUNG: Magnification his not closed/has remainder. Consider using a different magnification factor.")

magnify = torch.nn.AvgPool2d(kernel_size = magnification_factor)
# stride is by default the kernel size

# Explicit first dim
input = hr_scene_bed_tensor.unsqueeze(0)
lr_scene_bed_tensor = magnify(input)

print("LR tensor shape:", lr_scene_bed_tensor.shape)

HR tensor shape: torch.Size([1, 45, 45])
LR tensor shape: torch.Size([1, 9, 9])


In [20]:
fig = px.imshow(lr_scene_bed_tensor.squeeze(), 
                color_continuous_scale = 'RdBu_r',
                origin = "upper", 
                title = "LR bed scene")
fig.show()

In [40]:
fig = go.Figure(go.Heatmap(z = lr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'))
# fig.update_layout(yaxis_scaleanchor = "x")
# matrix display
fig.update_yaxes(autorange = "reversed")
fig.show()

In [79]:
fig = make_subplots(rows = 1, cols = 2,
                    subplot_titles = ("low res", "high res"))

fig.add_trace(go.Heatmap(z = lr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor.squeeze(), 
                           colorscale = 'RdBu_r'),
                           row = 1, col = 2)

fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()


In [94]:
fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("LR bed", "HR bed", "HR surface"))

fig.add_trace(go.Heatmap(z = lr_scene_bed_tensor.squeeze(), 
                           colorscale = 'haline'),
                           row = 1, col = 1)

fig.add_trace(go.Heatmap(z = hr_scene_bed_tensor.squeeze(), 
                           colorscale = 'haline'),
                           row = 1, col = 2)

fig.add_trace(go.Heatmap(z = hr_scene_sur_tensor.squeeze(), 
                           colorscale = 'gray'),
                           row = 1, col = 3)

fig.update_layout(autosize = False, height = 400, width = 900)
fig.update_layout(plot_bgcolor = 'rgba(0,0,0,0)')
# matrix style
fig.update_yaxes(autorange = "reversed")
fig.update_traces(showscale = False)
fig.show()

We will use HR surface to increase the resolution of the bedrock data.

In [121]:
from IPython.display import display, Math, Latex

display(Math(r'k\left(A_{H}, A_{H}^{\prime}\right)=\sigma_{f}^{2} k_{S}\left(A_{H}, A_{H}^{\prime}\right) k_{P}\left(P_{H}\left(A_{H}\right), P_{H}\left(A_{H}^{\prime}\right)\right)'))

<IPython.core.display.Math object>

In [184]:
display(Math(r'k_{P}\left(P_{H}\left(A_{H}\right), P_{H}\left(A_{H}^{\prime}\right)\right)=\exp \left\{-\frac{\left(P_{H}\left(A_{H}\right)-P_{H}\left(A_{H}^{\prime}\right)\right)^{2}}{2 \lambda_{P}^{2}}\right\}'))

<IPython.core.display.Math object>

$$ k_{P}: \ \text{Pixel intensity covariance function.}$$
$$ \text{Coupling/decoupling of pixels based on HR context.}$$
$$ \text{RBF (Squared exponential) kernel without output scale. Distance is calculated only over pixel values:
 We can flatten the image channel (or channels) as this has no spatial meaning}$$

In [151]:
mask = torch.zeros(size = hr_scene_bed_tensor.squeeze().shape)
mask[-1, :] = torch.ones(size = (hr_scene_bed_tensor.squeeze().shape[0],))

fig = go.Figure(go.Heatmap(z = hr_scene_bed_tensor.squeeze() * mask, colorscale = 'haline'))
fig.update_layout(autosize = False, height = 600, width = 600)
fig.update_yaxes(autorange = "reversed")
fig.show()

In [181]:
# Standardise to get sensitivity of default hp's
def NormalizeTensor(data):
    return (data - torch.min(data)) / (torch.max(data) - torch.min(data))
 
scaled_hr_scene_bed_tensor = NormalizeTensor(hr_scene_bed_tensor)

torch.Size([45])

In [362]:
kp_covar_module = gpytorch.kernels.RBFKernel()
# This kernel does not have an outputscale parameter - just like in paper

lazy_covar_matrix = kp_covar_module(scaled_hr_scene_bed_tensor[44, :]) # Returns a RootLinearOperator
kp_covar_matrix = lazy_covar_matrix.to_dense() # Gets the actual tensor for this kernel matrix

In [363]:
# Detach from computational graph
fig = go.Figure(go.Heatmap(z = kp_covar_matrix.detach(), colorscale = 'haline'))
fig.update_layout(autosize = False, height = 600, width = 600, title = "Covaraince matrix for last row of HR bed")
fig.update_yaxes(autorange = "reversed")
fig.show()

### Ks covariance term

In [328]:
# arange explude the last one
xs = torch.arange(1, 46).repeat(45,1)
ys = xs.T

# divide by 45 to standardise
mid_points = torch.cat((ys.unsqueeze(0), xs.unsqueeze(0)), dim = 0)/45

In [329]:
def ks_covariance_function(tensor, lambda_s = 0.2):
    """_summary_

    Args:
        tensor (_type_): _description_
        lambda_s (float, optional): hyperparamter, must be > 0. Defaults to 0.2.

    Returns:
        _type_: _description_
    """
    # If tensor is not flat, flatten
    if len(list(tensor.shape)) > 2:
        # 2 is hardcoded
        tensor = tensor.reshape(2, -1)

    ### Euclidean distance (2D) ###
    # Broadcast and calculate pairwise distances, 
    dist = tensor.unsqueeze(-1) - tensor.unsqueeze(-2)
    dist_sqr = torch.pow(dist, exponent = 2)
    # sum across x and y axis
    dist_sum = torch.sum(dist_sqr, dim = 0)
    # take sqrt
    euc_dist = torch.sqrt(dist_sum)

    Z = (euc_dist / lambda_s)

    # Mask large Z's with nan
    Z[Z >= 1] = float('nan')

    # first term pushes small distanced to 0 and distances near 1 close to zero
    # second terms is clipped at 1 so that smal dists will approach 1
    cov_matrix = torch.pow(1 - Z, exponent = 3) * ((3 * Z) + 1)
    cov_matrix[torch.isnan(cov_matrix)] = 0

    return cov_matrix

In [330]:
# Visualise mapping
Z = torch.tensor(np.linspace(0, 1, 100))
cov_matrix = torch.pow(1. - Z, exponent = 3) * ((3 * Z) + 1)

px.scatter(x = Z.numpy(), y = cov_matrix.numpy(), title = "Mapping of Z").update_layout(
    xaxis_title = "Z value", yaxis_title = "covariance"
)

In [334]:
mid_points_last_row = mid_points.reshape(2, -1)[:, -45:]

ks_covar_matrix = ks_covariance_function(mid_points_last_row)

torch.Size([45, 45])

In [335]:
fig = go.Figure(go.Heatmap(z = ks_covar_matrix.detach(), colorscale = 'haline'))
fig.update_layout(autosize = False, height = 600, width = 600, title = "Covaraince matrix for last row of HR bed")
fig.update_yaxes(autorange = "reversed")
fig.show()

In [442]:
def ks_kp_covar(lambda_s = 0.3, lambda_p = 0.4, sigma_f = 1.0, rowcol_index_tuple = (44, None), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor):
    # Subset rows/columns
    (row_index, column_index) = rowcol_index_tuple
    if (column_index == None):
        ks_input = mid_points[:, row_index, :]
        kp_input = kp_ds[row_index, :]
    else:
        ks_input = mid_points[:, :, column_index]
        kp_input = kp_ds[:, column_index]

    ### Ks ### smooth, sparse, local
    # lambda_s is the hp that controls the receptive field
    # ks input
    mid_points
    ks_covar_matrix = ks_covariance_function(ks_input, lambda_s = lambda_s)


    ### Kp ### coupling(decoupling) of similar(dissimilar) pixel values, non-stationary
    # lambda_p is the lengthscale of the RBF kernel
    # default 0.6931
    kp_covar_module = gpytorch.kernels.RBFKernel()
    kp_covar_module.lengthscale = lambda_p
    # This kernel does not have an outputscale parameter - just like in paper
    # ks input
    kp_covar_matrix = kp_covar_module(kp_input).to_dense()

    ### VIS ###
    # sigma_f is the output variance of the product term
    # Fix cmin and cmax to see sensitivity to output scalar
    fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("Ks", "Kp", "Ks * Kp * sigma_f"))

    fig.add_trace(go.Heatmap(z = ks_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                         row = 1, col = 1)

    fig.add_trace(go.Heatmap(z = kp_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                         # colorscale = 'haline'),
                         row = 1, col = 2)
    
    fig.add_trace(go.Heatmap(z = ks_covar_matrix.detach().numpy() * kp_covar_matrix.detach().numpy() * sigma_f,
                             # Trade off paramater (ks_covar_matrix.detach().numpy() * (1 - sigma_f)) * (kp_covar_matrix.detach().numpy() * sigma_f)
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                           row = 1, col = 3)

    fig.update_layout(autosize = False, height = 400, width = 900)
    fig.update_yaxes(autorange = "reversed") # matrix style
    fig.update_traces(showscale = False)
    fig.show()

In [460]:
ks_kp_covar(lambda_s = 0.6, lambda_p = 0.08, sigma_f = 0.8, rowcol_index_tuple = (None, 44), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)
# Good example:
# ks_kp_covar(lambda_s = 0.6, lambda_p = 0.08, sigma_f = 0.8, rowcol_index_tuple = (None, 44), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)
# higher lambda_s improves receptive spatial region. Cut-off point for zero covariance.
# lower lambda_p makes it more sensitive to filter out high pixel correlation values
# output scaling
# ks_kp_covar(lambda_s = 0.99, lambda_p = 0.8, sigma_f = 0.5, rowcol_index_tuple = (1, None), ks_ds = mid_points, kp_ds = scaled_hr_scene_bed_tensor)

This is more interesting if we look at covariance not with the same vector:

Use one row and one column vector.

In [441]:
fig = make_subplots(rows = 1, cols = 3,
                    subplot_titles = ("Ks", "Kp", "Product Ks * Kp"))

fig.add_trace(go.Heatmap(z = ks_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                         row = 1, col = 1)

fig.add_trace(go.Heatmap(z = kp_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                         # colorscale = 'haline'),
                         row = 1, col = 2)

fig.add_trace(go.Heatmap(z = ks_covar_matrix.detach().numpy() * kp_covar_matrix.detach().numpy(), 
                         zmin = 0, zmax = 1, 
                         colorscale = [[0, "rgb(242, 243, 34)"],
                                       [0.25, "rgb(251, 168, 54)"],
                                       [0.45, "rgb(240, 130, 77)"],
                                       [0.65, "rgb(202, 69, 122)"],
                                       [0.85, "rgb(136, 22, 163)"],
                                       [1, "rgb(43, 22, 146)"]]),
                           row = 1, col = 3)

fig.update_layout(autosize = False, height = 400, width = 900)
fig.update_yaxes(autorange = "reversed") # matrix style
fig.update_traces(showscale = False)
fig.show()

## Low res

In [483]:
begin{aligned}
& P_{*}\left(A_{H}\right)= \\
& \quad \mu+k\left(A_{H}, A_{L}\right)\left[k\left(A_{L}, A_{L}\right)+\sigma_{n}^{2} \mathbf{I}\right]^{-1}\left(P_{L}\left(A_{L}\right)-\mu\right) .
end{aligned}

SyntaxError: invalid syntax (1872837525.py, line 1)

In [462]:

begin{aligned}
& P_{*}\left(A_{H}\right)= \\
& \quad \mu+k\left(A_{H}, A_{L}\right)\left[k\left(A_{L}, A_{L}\right)+\sigma_{n}^{2} \mathbf{I}\right]^{-1}\left(P_{L}\left(A_{L}\right)-\mu\right) .
end{aligned}

SyntaxError: invalid syntax (3400846861.py, line 1)

Question: 

this definition of kP already supports
a multi-dimensional PH. (Euclidean?)

What about relationship between low res and high res image?